# Generador de datos - Heuristicos

Este notebook centraliza la generacion de datos experimentales para EA, PSO y DE.

Alcance de este notebook:
- ejecutar corridas reproducibles para los metodos heuristicos configurados
- guardar resultados limpios en archivos `json` y `csv`
- guardar un resumen consolidado por funcion para `analisis_resultados.ipynb`

Fuera de alcance aqui:
- graficas finales
- histogramas
- comparaciones visuales
- animaciones


In [5]:
import csv
import json
import sys
from pathlib import Path
from typing import Any, Callable

import numpy as np
from numpy.typing import NDArray

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from heuristicos import (
    run_differential_evolution,
    run_evolutionary_algorithm,
    run_particle_swarm_optimization,
)
from gradiente.funciones_objetivo import (
    goldstein_price,
    griewank,
    rastrigin,
    rosenbrock,
    schwefel,
    six_hump_camel,
)

Array = NDArray[np.float64]
ObjectiveFunction = Callable[[Array], float]

DATA_DIR = BASE_DIR / "datos"
MANIFEST_PATH = DATA_DIR / "manifest_heuristicos.json"

N_CORRIDAS_LISTA = [30]
SEED_BASE = 42

FUNCTION_CONFIGS: dict[str, dict[str, Any]] = {
    "rosenbrock": {
        "display_name": "Rosenbrock",
        "dimensions": [2, 3],
        "objective_function": rosenbrock,
        "bounds": (-2.048, 2.048),
        "known_optimum_by_dimension": {
            2: [1.0, 1.0],
            3: [1.0, 1.0, 1.0],
        },
        "heuristics": {
            "ea": {"population_size": 40, "max_iterations": 120, "elitism_fraction": 0.2, "mutation_fraction": 0.1},
            "pso": {"swarm_size": 40, "max_iterations": 120, "inertia_weight": 0.7, "cognitive_weight": 1.5, "social_weight": 1.5},
            "de": {"population_size": 40, "max_iterations": 120, "mutation_factor": 0.8, "crossover_rate": 0.7},
        },
    },
    "rastrigin": {
        "display_name": "Rastrigin",
        "dimensions": [2, 3],
        "objective_function": rastrigin,
        "bounds": (-5.12, 5.12),
        "known_optimum_by_dimension": {
            2: [0.0, 0.0],
            3: [0.0, 0.0, 0.0],
        },
        "heuristics": {
            "ea": {"population_size": 40, "max_iterations": 120, "elitism_fraction": 0.2, "mutation_fraction": 0.1},
            "pso": {"swarm_size": 40, "max_iterations": 120, "inertia_weight": 0.7, "cognitive_weight": 1.5, "social_weight": 1.5},
            "de": {"population_size": 40, "max_iterations": 120, "mutation_factor": 0.8, "crossover_rate": 0.7},
        },
    },
    "schwefel": {
        "display_name": "Schwefel",
        "dimensions": [2, 3],
        "objective_function": schwefel,
        "bounds": (-500.0, 500.0),
        "known_optimum_by_dimension": {
            2: [420.968746, 420.968746],
            3: [420.968746, 420.968746, 420.968746],
        },
        "heuristics": {
            "ea": {"population_size": 50, "max_iterations": 140, "elitism_fraction": 0.2, "mutation_fraction": 0.1},
            "pso": {"swarm_size": 50, "max_iterations": 140, "inertia_weight": 0.7, "cognitive_weight": 1.5, "social_weight": 1.5},
            "de": {"population_size": 50, "max_iterations": 140, "mutation_factor": 0.8, "crossover_rate": 0.7},
        },
    },
    "griewank": {
        "display_name": "Griewank",
        "dimensions": [2, 3],
        "objective_function": griewank,
        "bounds": (-600.0, 600.0),
        "known_optimum_by_dimension": {
            2: [0.0, 0.0],
            3: [0.0, 0.0, 0.0],
        },
        "heuristics": {
            "ea": {"population_size": 50, "max_iterations": 140, "elitism_fraction": 0.2, "mutation_fraction": 0.1},
            "pso": {"swarm_size": 50, "max_iterations": 140, "inertia_weight": 0.7, "cognitive_weight": 1.5, "social_weight": 1.5},
            "de": {"population_size": 50, "max_iterations": 140, "mutation_factor": 0.8, "crossover_rate": 0.7},
        },
    },
    "goldstein_price": {
        "display_name": "Goldstein-Price",
        "dimensions": [2],
        "objective_function": goldstein_price,
        "bounds": (-2.0, 2.0),
        "known_optimum_by_dimension": {
            2: [0.0, -1.0],
        },
        "heuristics": {
            "ea": {"population_size": 40, "max_iterations": 120, "elitism_fraction": 0.2, "mutation_fraction": 0.1},
            "pso": {"swarm_size": 40, "max_iterations": 120, "inertia_weight": 0.7, "cognitive_weight": 1.5, "social_weight": 1.5},
            "de": {"population_size": 40, "max_iterations": 120, "mutation_factor": 0.8, "crossover_rate": 0.7},
        },
    },
    "six_hump_camel": {
        "display_name": "Six-Hump Camel",
        "dimensions": [2],
        "objective_function": six_hump_camel,
        "bounds": (-3.0, 3.0),
        "known_optimum_by_dimension": {
            2: [0.089842, -0.712656],
        },
        "heuristics": {
            "ea": {"population_size": 40, "max_iterations": 120, "elitism_fraction": 0.2, "mutation_fraction": 0.1},
            "pso": {"swarm_size": 40, "max_iterations": 120, "inertia_weight": 0.7, "cognitive_weight": 1.5, "social_weight": 1.5},
            "de": {"population_size": 40, "max_iterations": 120, "mutation_factor": 0.8, "crossover_rate": 0.7},
        },
    },
}

METHOD_CONFIGS: dict[str, dict[str, Any]] = {
    "ea": {
        "runner": run_evolutionary_algorithm,
        "display_name": "Algoritmo evolutivo",
    },
    "pso": {
        "runner": run_particle_swarm_optimization,
        "display_name": "PSO",
    },
    "de": {
        "runner": run_differential_evolution,
        "display_name": "Evolucion diferencial",
    },
}


## Configuracion de ejecucion

Si un filtro se deja en `None`, se toman todos los valores configurados.

In [6]:
SELECTED_FUNCTIONS = None
SELECTED_DIMENSIONS = None
SELECTED_METHODS = None
SELECTED_N_CORRIDAS = None
SKIP_EXISTING_FILES = True
N_CORRIDAS_LISTA = [100, 500, 1000]


# Ejemplos utiles:
# SELECTED_FUNCTIONS = ["rosenbrock", "rastrigin"]
# SELECTED_DIMENSIONS = [2, 3]
# SELECTED_METHODS = ["ea", "pso"]
# SELECTED_N_CORRIDAS = [10]


In [7]:
def ensure_output_dirs() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)


def get_function_dir(function_name: str) -> Path:
    function_dir = DATA_DIR / function_name
    function_dir.mkdir(parents=True, exist_ok=True)
    return function_dir


def build_seed(function_name: str, method_name: str, dimension: int, run_index: int) -> int:
    key = f"{function_name}:{method_name}:{dimension}:{run_index}"
    checksum = sum(ord(char) for char in key)
    return SEED_BASE + checksum


def serialize_parameters(parameters: dict[str, Any]) -> str:
    return json.dumps(parameters, ensure_ascii=True, sort_keys=True)


def output_paths(function_name: str, method_name: str, dimension: int, n_runs: int) -> tuple[Path, Path]:
    base_name = f"{function_name}_{method_name}_{dimension}d_n{n_runs}"
    function_dir = get_function_dir(function_name)
    return function_dir / f"{base_name}.json", function_dir / f"{base_name}.csv"


def should_run(function_name: str, method_name: str, dimension: int, n_runs: int) -> bool:
    if SELECTED_FUNCTIONS is not None and function_name not in SELECTED_FUNCTIONS:
        return False
    if SELECTED_METHODS is not None and method_name not in SELECTED_METHODS:
        return False
    if SELECTED_DIMENSIONS is not None and dimension not in SELECTED_DIMENSIONS:
        return False
    if SELECTED_N_CORRIDAS is not None and n_runs not in SELECTED_N_CORRIDAS:
        return False
    return True


def files_already_exist(function_name: str, method_name: str, dimension: int, n_runs: int) -> bool:
    json_path, csv_path = output_paths(function_name, method_name, dimension, n_runs)
    return json_path.exists() and csv_path.exists()


def run_heuristic_experiment(
    *,
    function_name: str,
    display_name: str,
    method_name: str,
    method_display_name: str,
    runner: Callable[..., dict[str, Any]],
    objective_function: ObjectiveFunction,
    dimension: int,
    bounds: tuple[float, float],
    known_optimum: list[float],
    n_runs: int,
    parameters: dict[str, Any],
) -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    lower_bounds, upper_bounds = bounds
    optimum = np.asarray(known_optimum, dtype=float)

    for run_id in range(1, n_runs + 1):
        seed = build_seed(function_name, method_name, dimension, run_id)

        try:
            result = runner(
                objective_function=objective_function,
                dimension=dimension,
                lower_bounds=lower_bounds,
                upper_bounds=upper_bounds,
                seed=seed,
                **parameters,
            )
        except (OverflowError, FloatingPointError, ValueError) as error:
            print(
                f"Corrida omitida por error numerico: {function_name} | {method_name} | "
                f"{dimension}D | corrida={run_id} | error={error}"
            )
            continue

        best_solution = np.asarray(result["best_solution"], dtype=float)
        best_value = float(result["best_value"])
        iterations = int(result["iterations"])
        evaluations = int(result["evaluations"])

        if not np.isfinite(best_value) or not np.all(np.isfinite(best_solution)):
            print(
                f"Corrida omitida por resultado no finito: {function_name} | {method_name} | "
                f"{dimension}D | corrida={run_id}"
            )
            continue

        distance_to_optimum = float(np.linalg.norm(best_solution - optimum))

        best_values_history = np.asarray(result.get("best_values_history", []), dtype=float)

        results.append(
            {
                "funcion": function_name,
                "nombre_funcion": display_name,
                "corrida": run_id,
                "semilla": seed,
                "metodo": method_name,
                "nombre_metodo": method_display_name,
                "dimension": dimension,
                "limites": list(bounds),
                "optimo_conocido": optimum.tolist(),
                "mejor_solucion": best_solution.tolist(),
                "mejor_valor": best_value,
                "distancia_al_optimo": distance_to_optimum,
                "iteraciones": iterations,
                "evaluaciones": evaluations,
                "best_values_history": best_values_history.tolist(),
                "parametros": parameters.copy(),
            }
        )

    return results


def summarize_results(results: list[dict[str, Any]]) -> dict[str, Any]:
    if not results:
        raise ValueError("No hay resultados validos para resumir.")

    best_values = np.array([row["mejor_valor"] for row in results], dtype=float)
    distances = np.array([row["distancia_al_optimo"] for row in results], dtype=float)
    evaluations = np.array([row["evaluaciones"] for row in results], dtype=float)
    iterations = np.array([row["iteraciones"] for row in results], dtype=float)

    return {
        "n_corridas": len(results),
        "mejor_valor": float(best_values.min()),
        "peor_valor": float(best_values.max()),
        "promedio_valor": float(best_values.mean()),
        "mediana_valor": float(np.median(best_values)),
        "desviacion_valor": float(best_values.std(ddof=0)),
        "mejor_distancia_al_optimo": float(distances.min()),
        "promedio_distancia_al_optimo": float(distances.mean()),
        "promedio_evaluaciones": float(evaluations.mean()),
        "mediana_evaluaciones": float(np.median(evaluations)),
        "promedio_iteraciones": float(iterations.mean()),
        "mediana_iteraciones": float(np.median(iterations)),
    }


def save_run_results(
    *,
    function_name: str,
    method_name: str,
    dimension: int,
    n_runs: int,
    results: list[dict[str, Any]],
) -> tuple[Path, Path]:
    base_name = f"{function_name}_{method_name}_{dimension}d_n{n_runs}"
    function_dir = get_function_dir(function_name)
    json_path = function_dir / f"{base_name}.json"
    csv_path = function_dir / f"{base_name}.csv"

    with json_path.open("w", encoding="utf-8") as json_file:
        json.dump(results, json_file, indent=2, ensure_ascii=False)

    csv_rows = []
    for row in results:
        csv_rows.append(
            {
                "funcion": row["funcion"],
                "nombre_funcion": row["nombre_funcion"],
                "corrida": row["corrida"],
                "semilla": row["semilla"],
                "metodo": row["metodo"],
                "nombre_metodo": row["nombre_metodo"],
                "dimension": row["dimension"],
                "limites": json.dumps(row["limites"], ensure_ascii=True),
                "optimo_conocido": json.dumps(row["optimo_conocido"], ensure_ascii=True),
                "mejor_solucion": json.dumps(row["mejor_solucion"], ensure_ascii=True),
                "mejor_valor": row["mejor_valor"],
                "distancia_al_optimo": row["distancia_al_optimo"],
                "iteraciones": row["iteraciones"],
                "evaluaciones": row["evaluaciones"],
                "best_values_history": json.dumps(row["best_values_history"], ensure_ascii=True),
                "parametros": serialize_parameters(row["parametros"]),
            }
        )

    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=[
                "funcion",
                "nombre_funcion",
                "corrida",
                "semilla",
                "metodo",
                "nombre_metodo",
                "dimension",
                "limites",
                "optimo_conocido",
                "mejor_solucion",
                "mejor_valor",
                "distancia_al_optimo",
                "iteraciones",
                "evaluaciones",
                "best_values_history",
                "parametros",
            ],
        )
        writer.writeheader()
        writer.writerows(csv_rows)

    return json_path, csv_path


def save_function_summary(function_name: str, summary_rows: list[dict[str, Any]]) -> tuple[Path, Path]:
    function_dir = get_function_dir(function_name)
    json_path = function_dir / f"resumen_{function_name}.json"
    csv_path = function_dir / f"resumen_{function_name}.csv"

    with json_path.open("w", encoding="utf-8") as json_file:
        json.dump(summary_rows, json_file, indent=2, ensure_ascii=False)

    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=[
                "funcion",
                "nombre_funcion",
                "metodo",
                "nombre_metodo",
                "dimension",
                "n_corridas",
                "mejor_valor",
                "peor_valor",
                "promedio_valor",
                "mediana_valor",
                "desviacion_valor",
                "mejor_distancia_al_optimo",
                "promedio_distancia_al_optimo",
                "promedio_evaluaciones",
                "mediana_evaluaciones",
                "promedio_iteraciones",
                "mediana_iteraciones",
            ],
        )
        writer.writeheader()
        writer.writerows(summary_rows)

    return json_path, csv_path


def build_manifest() -> dict[str, Any]:
    return {
        "modulo": "heuristicos",
        "funciones": list(FUNCTION_CONFIGS.keys()),
        "metodos": list(METHOD_CONFIGS.keys()),
        "consumer_notebook": "analisis_resultados.ipynb",
        "descripcion": "Datos generados sin graficas ni animaciones.",
        "fecha_referencia": "2026-07-24",
        "filtros_ejecucion": {
            "selected_functions": SELECTED_FUNCTIONS,
            "selected_dimensions": SELECTED_DIMENSIONS,
            "selected_methods": SELECTED_METHODS,
            "selected_n_corridas": SELECTED_N_CORRIDAS,
            "skip_existing_files": SKIP_EXISTING_FILES
        },
        "directorios": {
            "datos_base": "heuristicos/datos"
        },
        "organizacion_corridas": "heuristicos/datos/<funcion>/<archivo>.json|csv",
        "organizacion_resumenes": "heuristicos/datos/<funcion>/resumen_<funcion>.json|csv",
        "campos_corridas": [
            "funcion",
            "nombre_funcion",
            "corrida",
            "semilla",
            "metodo",
            "nombre_metodo",
            "dimension",
            "limites",
            "optimo_conocido",
            "mejor_solucion",
            "mejor_valor",
            "distancia_al_optimo",
            "iteraciones",
            "evaluaciones",
            "best_values_history",
            "parametros"
        ],
        "campos_resumen": [
            "funcion",
            "nombre_funcion",
            "metodo",
            "nombre_metodo",
            "dimension",
            "n_corridas",
            "mejor_valor",
            "peor_valor",
            "promedio_valor",
            "mediana_valor",
            "desviacion_valor",
            "mejor_distancia_al_optimo",
            "promedio_distancia_al_optimo",
            "promedio_evaluaciones",
            "mediana_evaluaciones",
            "promedio_iteraciones",
            "mediana_iteraciones"
        ]
    }


def save_manifest() -> Path:
    manifest = build_manifest()
    with MANIFEST_PATH.open("w", encoding="utf-8") as json_file:
        json.dump(manifest, json_file, indent=2, ensure_ascii=False)
    return MANIFEST_PATH


def print_summary_row(row: dict[str, Any]) -> None:
    print(
        f"{row['funcion']} | {row['metodo']} | {row['dimension']}D | n={row['n_corridas']} | "
        f"mejor={row['mejor_valor']:.8f} | promedio={row['promedio_valor']:.8f} | "
        f"eval_prom={row['promedio_evaluaciones']:.2f}"
    )


In [8]:
ensure_output_dirs()

generated_anything = False
generated_labels: list[str] = []
function_summary_rows: dict[str, list[dict[str, Any]]] = {name: [] for name in FUNCTION_CONFIGS}

for function_name, config in FUNCTION_CONFIGS.items():
    for dimension in config["dimensions"]:
        known_optimum = config["known_optimum_by_dimension"][dimension]

        for method_name, method_config in METHOD_CONFIGS.items():
            parameters = config["heuristics"][method_name]

            for n_runs in N_CORRIDAS_LISTA:
                if not should_run(function_name, method_name, dimension, n_runs):
                    continue

                if SKIP_EXISTING_FILES and files_already_exist(function_name, method_name, dimension, n_runs):
                    print(f"Saltando existente: {function_name} | {method_name} | {dimension}D | n={n_runs}")
                    continue

                heuristic_results = run_heuristic_experiment(
                    function_name=function_name,
                    display_name=config["display_name"],
                    method_name=method_name,
                    method_display_name=method_config["display_name"],
                    runner=method_config["runner"],
                    objective_function=config["objective_function"],
                    dimension=dimension,
                    bounds=config["bounds"],
                    known_optimum=known_optimum,
                    n_runs=n_runs,
                    parameters=parameters,
                )

                save_run_results(
                    function_name=function_name,
                    method_name=method_name,
                    dimension=dimension,
                    n_runs=n_runs,
                    results=heuristic_results,
                )

                if not heuristic_results:
                    print(
                        f"Sin resultados validos: {function_name} | {method_name} | {dimension}D | n={n_runs}"
                    )
                    continue

                heuristic_summary = {
                    "funcion": function_name,
                    "nombre_funcion": config["display_name"],
                    "metodo": method_name,
                    "nombre_metodo": method_config["display_name"],
                    "dimension": dimension,
                    **summarize_results(heuristic_results),
                }

                generated_anything = True
                generated_labels.append(f"{function_name} | {method_name} | {dimension}D | n={n_runs}")
                function_summary_rows[function_name].append(heuristic_summary)
                print_summary_row(heuristic_summary)

if generated_anything:
    print()
    print("Guardando resumenes por funcion...")
    for function_name, rows in function_summary_rows.items():
        if rows:
            summary_json, summary_csv = save_function_summary(function_name, rows)
            print(f"- {function_name}: {summary_json.name}, {summary_csv.name}")
    print()
    print("Corridas nuevas generadas:")
    for label in generated_labels:
        print(f"- {label}")
else:
    print("No hubo nuevas corridas en esta ejecucion.")

manifest_path = save_manifest()
print()
print(f"Manifest guardado en: {manifest_path}")

rosenbrock | ea | 2D | n=100 | mejor=0.00198092 | promedio=0.06818983 | eval_prom=4840.00


KeyboardInterrupt: 